# 02 — Attach the monitors

Three system data metric functions, each on its own object so each can keep its own schedule. Every DMF on one object shares that object's `DATA_METRIC_SCHEDULE`.

| Check | Object | DMF | Expectation | Schedule |
| --- | --- | --- | --- | --- |
| Duplicate business key | `bt_orders` | `DUPLICATE_COUNT(order_id, line_no)` | `VALUE = 0` | `TRIGGER_ON_CHANGES` |
| Latest attempt volume | `bt_orders_latest_batch` | `ROW_COUNT()` | `800 <= VALUE AND VALUE <= 1200` | `5 MINUTE` |
| Latest attempt freshness | `bt_orders_latest_freshness` | `FRESHNESS(load_ts)` | `VALUE < 900` | `USING CRON 0 * * * * UTC` |

`TRIGGER_ON_CHANGES` is documented for DML on tables, so it is only set on `bt_orders`. The range uses `<=` because `BETWEEN` is not in the documented expectation operator list.

Scheduled evaluation is a different path from the proof. These schedules write rows to the account data-quality event table. This notebook does not read that table and does not wait for a schedule. The proof calls the checks directly.

**These schedules bill as Data Quality Monitoring until the teardown notebook drops the associations.** Suspending the warehouse does not stop them. The 5-minute row-count check is the standing cost, at 288 evaluations a day.

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

## Duplicate keys

Follows writes to the base table. The metric is the number of rows whose `(order_id, line_no)` value is duplicated.

In [ ]:
ALTER TABLE bt_orders SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

In [ ]:
ALTER TABLE bt_orders
  ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.DUPLICATE_COUNT
    ON (order_id, line_no)
    EXPECTATION no_duplicate_keys (VALUE = 0);

## Volume of the latest attempt

The view returns 0 rows when the ledger's newest batch deleted the table, which is exactly the case this check has to catch. A 5-minute cadence, not a cron, so it is not tied to the freshness view.

In [ ]:
ALTER VIEW bt_orders_latest_batch SET DATA_METRIC_SCHEDULE = '5 MINUTE';

In [ ]:
ALTER VIEW bt_orders_latest_batch
  ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT
    ON ()
    EXPECTATION rows_800_to_1200 (800 <= VALUE AND VALUE <= 1200);

## Freshness

On its own object so the hourly cron does not also pull the row-count check off its 5-minute schedule. The expectation is an age under 900 seconds.

In [ ]:
ALTER VIEW bt_orders_latest_freshness
  SET DATA_METRIC_SCHEDULE = 'USING CRON 0 * * * * UTC';

In [ ]:
ALTER VIEW bt_orders_latest_freshness
  ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS
    ON (load_ts)
    EXPECTATION fresher_than_900s (VALUE < 900);